In [1]:
import pygrib
import numpy as np
import sqlite3
import pandas as pd
import math

date,year-month,variable,point,value

591300

In [2]:
grbs = pygrib.open('data.grib')
# for grb in grbs:
#     print(grb)

In [ ]:
dir(grb)

In [12]:
#grb["year"],grb["month"],grb["day"],grb["hour"],grb["dataDate"],grb["dataTime"]
grb=grbs[1]
print(f'{grb["dataDate"]}{str(grb["dataTime"]).zfill(4)}')
# grb.values
# lats,lons=grb.latlons()

199001010000


In [ ]:
grb.data()

In [7]:
print(len(grb.data()[1]))
print(len(grb.data()[1][0]))


17
29


In [3]:

def decodeGrib(grb,records):
    dataTmp=grb.data()
    
    date=f'{grb["dataDate"]}{str(grb["dataTime"]).zfill(4)}'
    yrmo=f'{grb["year"]}{grb["month"]}'
    if yrmo not in data:
        data[yrmo]={}
    if date not in data[yrmo]:
        data[yrmo][date]={}
    tmp = np.empty((17,29))
    latslons= []
    for nrow,dat in enumerate(dataTmp[0]):
        # print(nrow,len(dat),data[0][nrow])
       ll=[]
       for ncol,val in enumerate(dat):
           val=dataTmp[0][nrow][ncol]
           pt=nlons*nrow+ncol
           var=grb['paramId']
           tmp=(yrmo,date,pt,var,val)
           records.append(tmp)
     #      tmp[nrow,ncol] = dataTmp[0][nrow][ncol]
        #    print(nrow,ncol,dataTmp[0][nrow][ncol],data[1][nrow][ncol],data[2][nrow][ncol])
      #      mm=[dataTmp[2][nrow][ncol],dataTmp[1][nrow][ncol]]
      #      ll.append(mm)
      #      pt=nlons*nrow+ncol
      #      if pt not in data[yrmo][date]:
      #          data[yrmo][date][pt]={}
      #      data[yrmo][date][pt][grb['paramId']]=dataTmp[0][nrow][ncol]
              
               
      # #     print(nrow,ncol,pt,dataTmp[2][nrow][ncol],dataTmp[1][nrow][ncol])
      #  latslons.append(ll)
    return records
        

In [4]:
nlats=17
nlons=29
data={}
records=[]
grbs = pygrib.open('data.grib')
for grb in grbs:
    if grb["year"] == 1990 and grb["month"] > 6:
        break
    records=decodeGrib(grb,records)
print("DONE")

DONE


In [21]:
records[500]

('19901', '199001010000', 7, 167, 265.241455078125)

In [22]:
connection = sqlite3.connect("Data/DB/era5_daily.db")

try:
    # Create a cursor object to interact with the database
    cursor = connection.cursor()
    
    # Create a table (if it doesn't already exist)
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS data (
            yrmo INTEGER ,
            date date NOT NULL,
            point INTEGER NOT NULL,
            variable integer not null,
            value  float
        )
    ''')
    
    # Insert records into the table
    cursor.executemany('INSERT INTO data (yrmo, date, point, variable, value) VALUES (?, ?, ?, ?, ?)', records)
    
    # Commit the transaction
    connection.commit()
    
    print("Records successfully written to the database.")
    
except sqlite3.Error as e:
    print(f"An error occurred: {e}")
    
finally:
    # Close the connection
    connection.close()

Records successfully written to the database.


In [2]:
connection = sqlite3.connect("Data/DB/era5_daily.db")
df= pd.read_sql("select * from data",connection)

In [6]:
df.head()

,yrmo,date,point,variable,value
0,19901,199001010000,0,168,262.890869
1,19901,199001010000,1,168,262.973633
2,19901,199001010000,2,168,262.699707
3,19901,199001010000,3,168,261.700684
4,19901,199001010000,4,168,260.437256


In [4]:

datesAll = list(df["date"].value_counts().to_dict().keys())
datesAll.sort()

In [5]:
datesAll[1]

199001010000

In [ ]:
def tTd2RH(t,td):
    rh=  100 * (math.exp((17.625 * td) / (243.04 + td))) / (math.exp((17.625 * t) / (243.04 + t)))
    return rh

def vpd(T,RH):
    tt=T-273.15
    es =0.6108*math.exp((17.27*tt)/(tt+237.3))
    if RH>1:
       val=100
    else:
        val=1
    ea=RH/val*es
    return es-ea


hist={}
ncount=0
for date in datesAll[1:]:
    ncount+=1
    tmp= df.loc[df["date"] == date]
    print(f"{ncount} of {len(datesAll)-1} : {date}") 
    pts=tmp["point"].value_counts().to_dict().keys()
    for pt in pts:
        t = float(tmp.loc[(tmp["point"] == pt) & (tmp["variable"] == 167),"value"].values.tolist()[0])
        td = float(tmp.loc[(tmp["point"] == pt) & (tmp["variable"] == 168),"value"].values.tolist()[0])
        yrmo = float(tmp.loc[(tmp["point"] == pt) & (tmp["variable"] == 168),"yrmo"].values.tolist()[0])
        rh=tTd2RH(t-273.15,td-273.15)
        vp=vpd(t,rh)
        
        if yrmo not in hist:
            hist[yrmo]={}
        if pt not in hist[yrmo]:
            hist[yrmo][pt]={}
            hist[yrmo][pt]["t"]=[]
            hist[yrmo][pt]["td"]=[]
            hist[yrmo][pt]["rh"]=[]
            hist[yrmo][pt]["vpd"]=[]
            
        hist[yrmo][pt]["t"].append(t)
        hist[yrmo][pt]["td"].append(td)
        hist[yrmo][pt]["rh"].append(rh)
        hist[yrmo][pt]["vpd"].append(vp)
        
                
       
    #    print(yrmo,pt,t,td,rh)
   
print("DONE") 

In [7]:
hist.keys()

dict_keys([19901.0, 19902.0, 19903.0, 19904.0, 19905.0, 19906.0])

In [10]:
pd.from_dict(hist)

AttributeError: module 'pandas' has no attribute 'from_dict'

In [30]:
for diff in range(0,30):
    for nn in range(2,3):
        tmean=sum(hist[19901.0][492]["t"][:nn])/len(hist[19901.0][492]["t"][:nn])
      #  tdmean=sum(hist[19901.0][492]["td"][:nn])/len(hist[19901.0][492]["td"][:nn])-diff
        rhmean=tTd2RH(tmean-273.15,tdmean-273.15)
        tdmean=tmean-diff
        rhmean=0
        tmean=0
        tdmean=0
        count=0
        for mm in range(nn):
           t= hist[19901.0][492]["t"][mm]-273.15
     #      td= hist[19901.0][492]["td"][mm]-273.15-diff
           td=t-diff
           rh=tTd2RH(t,td)
           rhmean+=rh
           count+=1
           tmean+=t
           tdmean+=td
            
        rhmean =rhmean/count
        tmean =tmean/count
        tdmean =tdmean/count
    
        rhmeanNew=tTd2RH(tmean,tdmean)
        
        print(nn,rhmean,rhmeanNew,rhmean-rhmeanNew,count,diff,tmean,tdmean)
        


2 100.0 100.0 0.0 2 0 -1.7901519775390398 -1.7901519775390398
2 92.87455166400272 92.87594649507567 -0.0013948310729432478 2 1 -1.7901519775390398 -2.7901519775390398
2 86.203664314094 86.2061388908129 -0.00247457671889606 2 2 -1.7901519775390398 -3.7901519775390398
2 79.96199768981216 79.96527679394433 -0.003279104132161592 2 3 -1.7901519775390398 -4.79015197753904
2 74.1253991552305 74.12924388519636 -0.0038447299658628253 2 4 -1.7901519775390398 -5.79015197753904
2 68.67086015161286 68.67506460327513 -0.004204451662275233 2 5 -1.7901519775390398 -6.79015197753904
2 63.576473691824745 63.5808618602124 -0.0043881683876563216 2 6 -1.7901519775390398 -7.79015197753904
2 58.821392892641434 58.82581578445263 -0.004422891811195484 2 7 -1.7901519775390398 -8.79015197753904
2 54.3857905403913 54.39012348736323 -0.004332946971935314 2 8 -1.7901519775390398 -9.79015197753904
2 50.250819684681396 50.25495984816229 -0.004140163480890635 2 9 -1.7901519775390398 -10.79015197753904
2 46.39857525427

In [25]:
tmean,tdmean

(-1.7901519775390398, -10.128256225585915)

In [ ]:



sumry={}
stop=False
for yrmo,dct in hist.items():
 #   print(yrmo,len(dct))
    for pt,dct2 in dct.items():
     #   print(pt,dct2.keys())
        t = [round(val,1) for val in dct2["t"]]
        td = [round(val,1) for val in dct2["td"]]
        rh = [round(val,1) for val in dct2["rh"]]
        vp=dct2["vpd"]
        tmean=round(sum(t)/len(t),1)
        tdmean=round(sum(td)/len(td),1)
        rhmean=round(sum(rh)/len(rh),1)
        vpdmean=round(sum(vp)/len(vp),1)
        
        rhmeanNew=tTd2RH(tmean-273.15,tdmean-273.15)
        vpdmeanNew=vpd(tdmean,rhmean)
        diff = abs(rhmean-rhmeanNew) 
    #     if diff > 10:
    #         print(yrmo,pt,tmean,tdmean,rhmean,rhmeanNew,diff)
    #         ttmp=t
    #         tdtmp=td
    #         rhtmp=rh
    #         hld=[]
    #         hld.append(rhmean)
    #         hld.append(rhmeanNew)
    #         stop=True
    #         break
    # if stop:
    #     break
    # tmean=sum(dct2["t"])/len(dct2["t"])
    # tdmean=sum(dct2["td"])/len(dct2["td"])
    # rhmean=sum(dct2["rh"])/len(dct2["rh"])
        print(tmean,tdmean,rhmean,tTd2RH(tmean-273.15,tdmean-273.15),vpdmean,vpdmeanNew,vpdmean-vpdmeanNew)
        if pt not in sumry:
            sumry[pt]=[]
            
        sumry[pt].append(abs(vpdmean-vpdmeanNew))

        

In [ ]:
sumry



In [102]:
sum(tdtmp)/len(tdtmp)
print(sum(rh)/len(rh))

70.86008064516133


In [103]:
tTd2RH(275.4-273.15,263.9-273.15)

42.358060966444576

In [105]:
min(ttmp),max(ttmp),min(tdtmp),max(tdtmp),min(rhtmp),max(rhtmp)

(262.6, 297.2, 254.6, 271.8, 25.9, 95.5)

In [ ]:
for nn,t in enumerate(ttmp):
    td=tdtmp[nn]
    rh=tTd2RH(t-273.15,td-273.15)
    print(t,td,rh,rhtmp[nn])

In [ ]:
for pt,dct in sumry.items():
    print(pt,sum(dct)/len(dct))
    

In [ ]:
for date,dct2 in data['19901'].items():
    
    td=dct2[1][168]
    t=dct2[1][167]
    print(date,t,td)

def tTd2RH(t,td):
    rh=  100 * (exp((17.625 * td) / (243.04 + td))) / (exp((17.625 * t) / (243.04 + t)))
    return rh
    

In [ ]:
def tTd2RH(t,td):
    rh=  100 * (exp((17.625 * td) / (243.04 + td))) / (exp((17.625 * t) / (243.04 + t)))
    return rh

tTd2RH(

In [10]:
grbs[1]["year"]

1990

In [27]:

min(list(data.keys()))
max(list(data.keys()))

'19921231900'

In [20]:
grb['parameterName'],grb['parameterUnits'],grb['paramIdECMF'],grb['paramId']

('2 metre dewpoint temperature', 'K', '168', 168)

In [ ]:
grb.keys()


In [24]:
grb['parameterName']

'2 metre dewpoint temperature'

In [ ]:
grb.keys()

In [ ]:
grb.keys()

In [ ]:
for grb in grbs:
    print(grb)

## netCdF

In [28]:
from netCDF4 import Dataset

# Open the NetCDF file
file_path = 'Data/ERA5/2m_temperature_0_daily-mean.nc'
dataset = Dataset(file_path, mode='r')

# Print metadata
print(dataset)

# Access variables
variables = dataset.variables
print(variables.keys())  # List of variables

# Access specific variable
# temperature = dataset.variables['temperature']
# print(temperature.shape)  # Dimensions of the variable
# print(temperature[:])     # Data array

# Close the dataset
dataset.close()

<class 'netCDF4.Dataset'>
root group (NETCDF4 data model, file format HDF5):
    GRIB_centre: ecmf
    GRIB_centreDescription: European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre: 0
    Conventions: CF-1.7
    institution: European Centre for Medium-Range Weather Forecasts
    history: 2024-12-15T00:29 GRIB to CDM+CF via cfgrib-0.9.14.1/ecCodes-2.36.0 with {"source": "2m_temperature.grib", "filter_by_keys": {"stream": ["oper"]}, "encode_cf": ["parameter", "time", "geography", "vertical"]}
earthkit.transforms.aggregate.temporal.daily_reduce(2m_temperature_0, how=mean, **{'time_shift': {'hours': 0}, 'remove_partial_periods': True})
    dimensions(sizes): valid_time(70), latitude(17), longitude(28)
    variables(dimensions): float32 t2m(valid_time, latitude, longitude), int64 number(), float64 latitude(latitude), float64 longitude(longitude), int64 valid_time(valid_time)
    groups: 
dict_keys(['t2m', 'number', 'latitude', 'longitude', 'valid_time'])


In [30]:
dataset[0]

TypeError: expected str, bytes or os.PathLike object, not int

In [ ]:
dir(dataset)

In [2]:
import xarray as xr

# Open the NetCDF file
file_path = 'Data/ERA5/2m_temperature_0_daily-mean.nc'
dataset = xr.open_dataset(file_path)

# Print dataset metadata
print(dataset)

# Access variables
# temperature = dataset['temperature']
# print(temperature)

libpoppler.so.140: cannot open shared object file: No such file or directory
libpoppler.so.140: cannot open shared object file: No such file or directory
libpoppler.so.140: cannot open shared object file: No such file or directory
libpoppler.so.140: cannot open shared object file: No such file or directory


<xarray.Dataset> Size: 134kB
Dimensions:     (valid_time: 70, latitude: 17, longitude: 28)
Coordinates:
    number      int64 8B ...
  * latitude    (latitude) float64 136B 41.0 40.75 40.5 ... 37.5 37.25 37.0
  * longitude   (longitude) float64 224B -109.0 -108.8 -108.5 ... -102.5 -102.2
  * valid_time  (valid_time) datetime64[ns] 560B 2024-10-01 ... 2024-12-09
Data variables:
    t2m         (valid_time, latitude, longitude) float32 133kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2024-12-15T00:29 GRIB to CDM+CF via cfgrib-0.9.1...


In [ ]:
dir(dataset)

In [8]:
lats=dataset.latitude[:]
lons=dataset.longitude[:]

In [10]:
lats[0:16]

<xarray.DataArray 'latitude' (latitude: 16)> Size: 128B
array([41.  , 40.75, 40.5 , 40.25, 40.  , 39.75, 39.5 , 39.25, 39.  , 38.75,
       38.5 , 38.25, 38.  , 37.75, 37.5 , 37.25])
Coordinates:
    number    int64 8B ...
  * latitude  (latitude) float64 128B 41.0 40.75 40.5 40.25 ... 37.75 37.5 37.25
Attributes:
    units:             degrees_north
    standard_name:     latitude
    long_name:         latitude
    stored_direction:  decreasing

In [11]:
len(lats)

17

In [12]:
len(lons)

28

In [13]:
dataset.values

<bound method Mapping.values of <xarray.Dataset> Size: 134kB
Dimensions:     (valid_time: 70, latitude: 17, longitude: 28)
Coordinates:
    number      int64 8B ...
  * latitude    (latitude) float64 136B 41.0 40.75 40.5 ... 37.5 37.25 37.0
  * longitude   (longitude) float64 224B -109.0 -108.8 -108.5 ... -102.5 -102.2
  * valid_time  (valid_time) datetime64[ns] 560B 2024-10-01 ... 2024-12-09
Data variables:
    t2m         (valid_time, latitude, longitude) float32 133kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2024-12-15T00:29 GRIB to CDM+CF via cfgrib-0.9.1...>

In [ ]:
variables=["2m_temperature","2m_dewpoint_temperature","10m_u_component_of_wind","10m_v_component_of_wind","surface_pressure"]

In [49]:
for start in range(2022,2022+1,4):
    end=start+3
    ofile=f"{variable}_{start}_{end}"
    years=[str(year) for year in range(start,end+1)]
    print(ofile,years)

2m_temperature_2022_2025 ['2022', '2023', '2024', '2025']


## Get Data

In [2]:
import cdsapi
variables=["2m_temperature","2m_dewpoint_temperature","10m_u_component_of_wind","10m_v_component_of_wind","surface_pressure"]


dataset = "reanalysis-era5-land"
variable= variables[0]
for start in range(2018,2022+1,4):
    end=start+3
    ofile=f"{variable}_{start}_{end}"
    years=[str(year) for year in range(start,end+1)]
    print(ofile,years)
    request = {
        "variable": [
             f"{variable}"
        ],
        "year": years,
        "month": [
            "01" , "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12"
           ],
        "day": [
            "01", "02", "03",
            "04", "05", "06",
            "07", "08", "09",
            "10", "11", "12",
            "13", "14", "15",
            "16", "17", "18",
            "19", "20", "21",
            "22", "23", "24",
            "25", "26", "27",
            "28", "29", "30",
            "31"
        ],
        "time": [
            "00:00", "03:00", "06:00",
            "09:00", "12:00", "15:00",
            "18:00", "21:00"
        ],
        "data_format": "grib",
        "download_format": "unarchived",
        "target":ofile,
        "area": [41.1, -109, 37, -102]
    }
    
    client = cdsapi.Client()
    client.retrieve(dataset, request).download()
    print(f"File {ofile} Downloaded")


2m_temperature_2018_2021 ['2018', '2019', '2020', '2021']


2024-12-28 12:53:33,660 INFO [2024-09-28T00:00:00] **Welcome to the New Climate Data Store (CDS)!** This new system is in its early days of full operations and still undergoing enhancements and fine tuning. Some disruptions are to be expected. Your 
[feedback](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) is key to improve the user experience on the new CDS for the benefit of everyone. Thank you.
2024-12-28 12:53:33,660 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2024-12-28 12:53:33,661 INFO [2024-09-16T00:00:00] Remember that you need to have an ECMWF account to use the new CDS. **Your old CDS credentials will not work in new CDS!**
2024-12-28 12:53:33,662 WARNING [2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using t

File 2m_temperature_2018_2021 Downloaded
2m_temperature_2022_2025 ['2022', '2023', '2024', '2025']


2024-12-28 14:04:57,687 INFO [2024-09-28T00:00:00] **Welcome to the New Climate Data Store (CDS)!** This new system is in its early days of full operations and still undergoing enhancements and fine tuning. Some disruptions are to be expected. Your 
[feedback](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) is key to improve the user experience on the new CDS for the benefit of everyone. Thank you.
2024-12-28 14:04:57,688 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2024-12-28 14:04:57,688 INFO [2024-09-16T00:00:00] Remember that you need to have an ECMWF account to use the new CDS. **Your old CDS credentials will not work in new CDS!**
2024-12-28 14:04:57,689 WARNING [2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using t

File 2m_temperature_2022_2025 Downloaded


In [52]:
! ls -lrt

total 1292512
-rw-r--r-- 1 joe joe   117018 Jun 14  2024  610prcp_20240613.zip
-rw-r--r-- 1 joe joe   108169 Jun 14  2024  814prcp_latest.zip
-rw-r--r-- 1 joe joe    50331 Jun 14  2024  Untitled.ipynb
-rw-r--r-- 1 joe joe       72 Jul  3 20:07  Untitled1.ipynb
-rw-r--r-- 1 joe joe       95 Jul  6 23:03  hld
-rw-r--r-- 1 joe joe    33216 Jul 19 17:09  decode_grib.ipynb
drwxr-xr-x 2 joe joe     4096 Jul 19 22:44  Boundaries
-rw-r--r-- 1 joe joe     5039 Jul 23 18:25  list
-rw-r--r-- 1 joe joe    13472 Jul 24 11:36  compare.ipynb
-rw-r--r-- 1 joe joe      461 Jul 25 12:55  test.py
-rw-r--r-- 1 joe joe        0 Jul 27 18:36  download-fails.txt
-rw-r--r-- 1 joe joe      841 Jul 28 21:46  temps.sql
-rw-r--r-- 1 joe joe    14198 Aug  2 16:03  02_Accessing_Database_Using_SQLite.ipynb
-rw-r--r-- 1 joe joe   409600 Aug  3 18:27  flxf.01.2017050400.201708.avrg.grib.grb2azk4dsgn.tmp
-rw-r--r-- 1 joe joe   745472 Aug  7 23:21  flxf.01.2020041800.202010.avrg.grib.grb28hphrs7i.tmp
-rw-r--r-- 1 joe jo

In [36]:
variables=["2m_temperature","2m_dewpoint_temperature","10m_u_component_of_wind","10m_v_component_of_wind","surface_pressure"]

for variable in variables:
    for start in range(1990,2020+1,5):
        end=start+4
        print(year)
        print([str(year) for year in range(start,end+1)])

2020
['1990', '1991', '1992', '1993', '1994']
2020
['1995', '1996', '1997', '1998', '1999']
2020
['2000', '2001', '2002', '2003', '2004']
2020
['2005', '2006', '2007', '2008', '2009']
2020
['2010', '2011', '2012', '2013', '2014']
2020
['2015', '2016', '2017', '2018', '2019']
2020
['2020', '2021', '2022', '2023', '2024']


In [41]:
ofile="2m_temperature_1990_1993.grb"
request = {
    "variable": [
         "2m_temperature"
    ],
    "year": ["1990"],
    "month": [
        "01"  
       ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "03:00", "06:00",
        "09:00", "12:00", "15:00",
        "18:00", "21:00"
    ],
    "data_format": "grib",
    "download_format": "unarchived",
    "target":ofile,
    "area": [41.1, -109, 37, -102]
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()
print(f"File {ofile} Downloaded")

2024-12-21 09:39:10,263 INFO [2024-09-28T00:00:00] **Welcome to the New Climate Data Store (CDS)!** This new system is in its early days of full operations and still undergoing enhancements and fine tuning. Some disruptions are to be expected. Your 
[feedback](https://jira.ecmwf.int/plugins/servlet/desk/portal/1/create/202) is key to improve the user experience on the new CDS for the benefit of everyone. Thank you.
2024-12-21 09:39:10,265 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.
2024-12-21 09:39:10,267 INFO [2024-09-16T00:00:00] Remember that you need to have an ECMWF account to use the new CDS. **Your old CDS credentials will not work in new CDS!**
2024-12-21 09:39:10,268 WARNING [2024-06-16T00:00:00] CDS API syntax is changed and some keys or parameter names may have also changed. To avoid requests failing, please use the "Show API request code" tool on the dataset Download Form to check you are using t

File 2m_temperature_1990_1993.grb Downloaded


In [39]:
request = {
    "variable": [
         "2m_temperature",
        "2m_dewpoint_temperature",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "surface_pressure"
    ],
    "year": ["1990"],
    "month": [
        "01"  "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
       ],
    "day": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12",
        "13", "14", "15",
        "16", "17", "18",
        "19", "20", "21",
        "22", "23", "24",
        "25", "26", "27",
        "28", "29", "30",
        "31"
    ],
    "time": [
        "00:00", "03:00", "06:00",
        "09:00", "12:00", "15:00",
        "18:00", "21:00"
    ],
    "data_format": "grib",
    "download_format": "unarchived",
    "target":"somefilename.grb",
    "area": [41.1, -109, 37, -102]
}

      "2m_temperature",
        "10m_u_component_of_wind",
        "10m_v_component_of_wind",
        "surface_pressure"

In [2]:
fin=open("listTd")
files=fin.readlines()

In [7]:
files[0]

'3b6d2d6585343df1fa6b220d9a38bff1.grib\n'

In [6]:
invt={}
for file in files:
    file=file.strip()
    print(file)
    grbs = pygrib.open(file)
    years={}
    months={}
    for grb in grbs:
       yr=grb["year"]
       mo=grb["month"]
       dy=grb["day"]
       var=grb['paramId']
       if yr not in years:
           years[yr]=0
       if mo not in months:
           months[mo]=0
       months[mo]+=1
       years[yr]+=1
       if var not in invt:
           invt[var]={}
       if yr not in invt[var]:
           invt[var][yr]={}
       if mo not in invt[var][yr]:
           invt[var][yr][mo]=0

       invt[var][yr][mo]+=1
    for yr in sorted(years.keys()):
        print(f"   {yr}  {years[yr]:7d}")
    for mo in months:
        print(f"     {mo}  {months[mo]:5d}")
    grbs.close()
print("DONE")

966fb7d78b4d2e417520a2f7958b93c2.grib
   1990     2448
   1991     2448
   1992     2448
   1993     2448
     2      4
     3    992
     4    960
     5    992
     6    960
     7    992
     8    992
     9    960
     10    992
     11    960
     12    988
cfb2098b0f78d06dd482797a87823c4b.grib
   1994     2448
   1995     2448
   1996     2448
   1997     2448
     2      4
     3    992
     4    960
     5    992
     6    960
     7    992
     8    992
     9    960
     10    992
     11    960
     12    988
48fc8ce7d183c8038856997043e4d21c.grib
   1998     2448
   1999     2448
   2000     2448
   2001     2448
     2      4
     3    992
     4    960
     5    992
     6    960
     7    992
     8    992
     9    960
     10    992
     11    960
     12    988
7c11c79397b0ce98bac2c9387e8f421c.grib
   2002     2448
   2003     2448
   2004     2448
   2005     2448
     2      4
     3    992
     4    960
     5    992
     6    960
     7    992
     8    992
     9 

In [4]:
for var in invt.keys():
    for year in range(1990,2025):
    #    year=str(year)
    
        print(f"{var:3d} {year:4d}",end="")
        for month in range(1,13):
            if month in invt[var][year]:
                cnt=invt[var][year][month]
            else:
                cnt=0
            print(f"{cnt:5d}",end="")
        print("")
        

168 1990    0    1  248  240  248  240  248  248  240  248  240  247
168 1991    0    1  248  240  248  240  248  248  240  248  240  247
168 1992    0    1  248  240  248  240  248  248  240  248  240  247
168 1993    0    1  248  240  248  240  248  248  240  248  240  247
168 1994    0    1  248  240  248  240  248  248  240  248  240  247
168 1995    0    1  248  240  248  240  248  248  240  248  240  247
168 1996    0    1  248  240  248  240  248  248  240  248  240  247
168 1997    0    1  248  240  248  240  248  248  240  248  240  247
168 1998    0    1  248  240  248  240  248  248  240  248  240  247
168 1999    0    1  248  240  248  240  248  248  240  248  240  247
168 2000    0    1  248  240  248  240  248  248  240  248  240  247
168 2001    0    1  248  240  248  240  248  248  240  248  240  247
168 2002    0    1  248  240  248  240  248  248  240  248  240  247
168 2003    0    1  248  240  248  240  248  248  240  248  240  247
168 2004    0    1  248  240  248 

In [5]:
grb

7232:2 metre dewpoint temperature:K (instant):regular_ll:surface:level 0:fcst time 21 hrs:from 202412170000